In [1]:
# =========================================
# IMPORTS (already installed in yolo_env)
# =========================================

from ultralytics import YOLO
import cv2
import numpy as np

FRAME_SKIP = 4
frame_index = 0


# =========================================
# PATHS (UPDATE FOR YOUR SYSTEM)
# =========================================

VIDEO_PATH = "Testing_video1.mp4"        # input video
AUTO_MODEL_PATH = "best (1).pt"           # final trained auto model
COCO_MODEL_PATH = "yolov8s.pt"             # COCO model
OUTPUT_VIDEO = "final_output_with_ids.mp4"

# =========================================
# PARAMETERS
# =========================================

CONF_MARGIN = 0.15
IOU_MATCH = 0.5
COCO_VEHICLES = {"car", "bus", "truck", "motorcycle"}

# =========================================
# LOAD MODELS
# =========================================

coco_model = YOLO(COCO_MODEL_PATH)
auto_model = YOLO(AUTO_MODEL_PATH)

# =========================================
# IOU FUNCTION
# =========================================

def iou(a, b):
    xA = max(a[0], b[0])
    yA = max(a[1], b[1])
    xB = min(a[2], b[2])
    yB = min(a[3], b[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (a[2] - a[0]) * (a[3] - a[1])
    areaB = (b[2] - b[0]) * (b[3] - b[1])
    return inter / (areaA + areaB - inter + 1e-6)

# =========================================
# VIDEO SETUP
# =========================================

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(
    OUTPUT_VIDEO,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (W, H)
)

print("🚀 Running vehicle detection + tracking pipeline...")

# =========================================
# MAIN LOOP
# =========================================

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_index += 1

    # ⏩ FRAME SKIPPING
    if frame_index % FRAME_SKIP != 0:
        out.write(frame)   # still write frame to output video
        continue


    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # ===== COCO DETECTION + TRACKING =====
    coco_res = coco_model.track(
        frame_rgb,
        persist=True,
        conf=0.25,
        verbose=False
    )[0]

    vehicle_boxes = []

    if coco_res.boxes is not None:
        for b in coco_res.boxes:
            label = coco_res.names[int(b.cls)]
            conf = float(b.conf)
            track_id = int(b.id) if b.id is not None else None

            if label in COCO_VEHICLES:
                x1, y1, x2, y2 = map(int, b.xyxy[0])
                vehicle_boxes.append((x1, y1, x2, y2, label, conf, track_id))

    # ===== CROP VEHICLES FOR AUTO MODEL =====
    crops = []
    crop_coords = []

    for (x1, y1, x2, y2, _, _, _) in vehicle_boxes:
        crop = frame_rgb[y1:y2, x1:x2]
        if crop.size > 0:
            crops.append(crop)
            crop_coords.append((x1, y1))

    auto_results = []
    if len(crops) > 0:
        auto_results = auto_model.predict(crops, conf=0.25, verbose=False)

    tracked_autos = []

    for i, res in enumerate(auto_results):
        if res.boxes is None:
            continue
        xoff, yoff = crop_coords[i]
        for b in res.boxes:
            ax1, ay1, ax2, ay2 = b.xyxy[0]
            tracked_autos.append((
                int(xoff + ax1),
                int(yoff + ay1),
                int(xoff + ax2),
                int(yoff + ay2),
                "auto",
                float(b.conf)
            ))

    # ===== CONFIDENCE FUSION =====
    final_detections = [
        [x1, y1, x2, y2, l, c, tid]
        for (x1, y1, x2, y2, l, c, tid) in vehicle_boxes
    ]

    for (ax1, ay1, ax2, ay2, alabel, aconf) in tracked_autos:
        best_iou = 0
        best_idx = -1

        for i, (cx1, cy1, cx2, cy2, clabel, cconf, tid) in enumerate(final_detections):
            overlap = iou((ax1, ay1, ax2, ay2), (cx1, cy1, cx2, cy2))
            if overlap > best_iou:
                best_iou = overlap
                best_idx = i

        if best_iou >= IOU_MATCH:
            coco_conf = final_detections[best_idx][5]
            if aconf >= coco_conf + CONF_MARGIN:
                final_detections[best_idx][4] = "auto"
                final_detections[best_idx][5] = aconf
        else:
            final_detections.append([ax1, ay1, ax2, ay2, "auto", aconf, None])

    # ===== DRAW RESULTS =====
    for (x1, y1, x2, y2, label, conf, tid) in final_detections:
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)

        id_text = f"ID {tid}" if tid is not None else "ID N/A"
        cv2.putText(
            frame,
            f"{label.upper()} {conf:.2f} | {id_text}",
            (x1, max(30, y1 - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.0,
            (0, 255, 0),
            3
        )

    out.write(frame)

cap.release()
out.release()

print("✅ Done. Output saved as:", OUTPUT_VIDEO)

🚀 Running vehicle detection + tracking pipeline...
✅ Done. Output saved as: final_output_with_ids.mp4
